In [1]:
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam  #this allows us to provide custome learning rate like "Adam(learning_rate = 0.0001)" 
# without importing this custom class , keras allow us to use it like model.compile(optimize = 'adam', loss = , metrics =), but doesnt let us 
# mention the learning rate inside

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
''' CONVENTIONS USED '''
# N = num of samples 
# T = time steps / lags / sequence length 
# D = dimensions / num of time series / diff time series / num of features / embedding dimension of each x(t) 
# M = output size of each hidden unit h(t)
# K = output size of single output vectors y(t)

' CONVENTIONS USED '

In [70]:
# making some data
N = 1
T = 10
D = 3
# M = 
K = 2
X = np.random.randn(N, T, D)

In [71]:
# make rnn
M = 5 
i = Input(shape = (T,D)) #10x3
x = SimpleRNN(M)(i) # many to one - 1 hidden output of size M = 5
x = Dense(K)(x) # 1 output of shape 2 - since N =2 , shape will be 1x2

model = Model(i,x)

In [72]:
# random weights and data are random but just checking the shape
Yhat = model.predict(X)
print(Yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 237ms/step
[[-0.8508244 -0.6943974]]


In [73]:
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 10, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 2)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57 (228.00 B)

 Trainable params: 57 (228.00 B)

 Non-trainable params: 0 (0.00 B)

In [74]:
# num of parameters working - W of hidden will be [wh, wx], wh is for applying it on hidden unit h(t-1) and wx for applying on x(t).
# since the output of each hidden unit h(t) is 5, then wh will be of shape (5,5), and Wx for x(t) will be of shape (3,5)
# so num of parametrs in W will be 5*5 + 3*5 = 40 parameters plus 5(M) for bias , so total parameters will be 40+5 = 45 
# for output layer , input is M vectors (many to one) and output is 2, so num of parameters will be 5*2 = 10 and 2 additonal bias terms, making
# total parameter count being 12

In [75]:
# SIMPLE RNN LAYER
model.layers[1].get_weights()

[array([[-0.32066202,  0.8228548 , -0.38077393, -0.83890265,  0.48279518],
        [-0.73762673, -0.3504727 , -0.24349076,  0.40708023,  0.41192418],
        [-0.02972186, -0.6835819 , -0.05424345,  0.0065257 , -0.33818316]],
       dtype=float32),
 array([[ 0.11185265,  0.20520654, -0.5637648 , -0.27590224, -0.742581  ],
        [ 0.161491  , -0.7187317 , -0.06121033,  0.57952815, -0.34314126],
        [ 0.7161716 ,  0.46326792, -0.1538984 ,  0.46523064,  0.17988002],
        [ 0.17680436, -0.39863074, -0.6623162 , -0.2994568 ,  0.53056264],
        [-0.6459475 ,  0.2603677 , -0.4648388 ,  0.5309532 ,  0.13028458]],
       dtype=float32),
 array([0., 0., 0., 0., 0.], dtype=float32)]

In [76]:
# SIMPLE RNN LAYER
print(model.layers[1].get_weights()[0].shape) # WX

print(model.layers[1].get_weights()[1].shape) # WH

print(model.layers[1].get_weights()[2].shape) # B

(3, 5)
(5, 5)
(5,)


In [77]:
a, b, c = model.layers[1].get_weights()
print(a.shape, b.shape, c.shape)

(3, 5) (5, 5) (5,)


In [78]:
# OUTPUT DENSE LAYER
model.layers[2].get_weights()

[array([[-0.7846662 , -0.79385746],
        [ 0.07143509, -0.39346266],
        [-0.2545132 , -0.2601611 ],
        [-0.571277  , -0.84794074],
        [ 0.4227016 ,  0.21151888]], dtype=float32),
 array([0., 0.], dtype=float32)]

In [79]:
print(model.layers[2].get_weights()[0].shape) # WY
print(model.layers[2].get_weights()[1].shape) # B

(5, 2)
(2,)


In [80]:
Wx, Wh, bh = model.layers[1].get_weights()
Wo, bo = model.layers[2].get_weights()

In [81]:
X[0]

array([[-1.80097726,  0.19082084,  2.2631403 ],
       [-1.10560256, -1.24726983,  0.75172103],
       [ 1.41247021,  1.72509641, -0.17722517],
       [-1.78659106, -0.01527027,  0.5719531 ],
       [-0.98441363, -0.72114754, -0.4017107 ],
       [-0.14474133, -0.45075721, -0.79043025],
       [ 0.76339979, -0.17285095,  1.12245903],
       [ 1.06930576, -1.14586624, -1.05210766],
       [ 0.63978002,  0.86638155, -1.43434011],
       [-0.92407915, -0.07307919, -0.65187272]])

In [82]:
X[0].shape

(10, 3)

In [83]:
X[0][0].shape #looping through each time step in X[0]

(3,)

In [84]:
print(Wx.shape, Wh.shape, Wo.shape)
print(bh.shape, bo.shape)

(3, 5) (5, 5) (5, 2)
(5,) (2,)


In [86]:
# BUILDING RNN FROM MANUALLY
h_last = np.zeros(M) # initial hidden state dimension
x = X[0] # X only has 1 sample data - just getting that
Yhats = [] #for storing the output

for t in range(T):
    h = np.tanh( x[t].dot(Wx) + h_last.dot(Wh) + bh)
    y = h.dot(Wo) + bo # only care about the last value 
    Yhats.append(y)

    h_last = h

print(Yhats[-1])

[-0.85082442 -0.69439738]


In [88]:
model.predict(X)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step


array([[-0.8508244, -0.6943974]], dtype=float32)

In [87]:
# results from both manual model and library model matches exactly